Test A: SMS reminders (yes/no outcome) <br>

Step 1: Plan. <br>
• H₀: the % of missed appointments is the same with and without SMS. <br>
• H₁: the % is different. <br>
• Starting value (from records): 25% missed. Smallest change worth catching: down to 20%. The 0.05 line, power 80%.

In [1]:
import numpy as np                          # numpy: tools for numbers and random numbers
import pandas as pd
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize


In [2]:
# Sample-size calculator: how many patients do we need in each group?
size = proportion_effectsize(0.25, 0.20)    # starting value 25%, hoped-for 20%
n = NormalIndPower().solve_power(effect_size=size, alpha=0.05, power=0.8)
print(round(n))                             # prints 1092 → we round up to 1,100 per group

1092


Step 2: Get the data (practice data). 2,200 patients. Each one is put in the SMS group or the no-SMS group at random. Then, for each patient, the computer "flips a weighted coin" to decide if they miss: a 20% chance for SMS patients and 25% for the others.

In [3]:
rng = np.random.default_rng(42)             # random number maker; 42 makes your numbers match these notes

patients_per_group = 1100
# A list of 2,200 labels: 1,100 "no_sms" and 1,100 "sms"
group = np.array(["no_sms"] * patients_per_group + ["sms"] * patients_per_group)
rng.shuffle(group)                          # shuffle the labels = random assignment

# Each patient's chance of missing: 0.20 if in the sms group, otherwise 0.25
chance_of_missing = np.where(group == "sms", 0.20, 0.25)

# Weighted coin: a random number between 0 and 1. Below the patient's chance → "yes" (missed)
missed = np.where(rng.random(len(group)) < chance_of_missing, "yes", "no")

df = pd.DataFrame({"group": group, "missed": missed})   # put the two lists into a table
print(df.head())                                          # first 5 patients

    group missed
0  no_sms     no
1  no_sms    yes
2     sms     no
3  no_sms     no
4  no_sms     no


Step 3: Test.

In [4]:
table = pd.crosstab(df["group"], df["missed"])           # count table
print(table)
#          no   yes
# no_sms  819   281
# sms     882   218

# % missed in each group = "yes" ÷ everyone in that row × 100
percent_missed = table["yes"] / table.sum(axis=1) * 100
print(percent_missed.round(1))              # no_sms 25.5, sms 19.8

print(stats.chi2_contingency(table).pvalue) # prints 0.0016

missed   no  yes
group           
no_sms  819  281
sms     882  218
group
no_sms    25.5
sms       19.8
dtype: float64
0.001596955549566989


<b>Step 4: Decide. </b><br>
• p = 0.0016, below 0.05, so we reject H₀. The difference is real. <br>
• Size of the effect: 25.5% → 19.8%, a drop of 5.7 points. That is more than the 5 points the manager said was worth paying for. <br>
• Decision: "SMS reminders reduced missed appointments from 25.5% to 19.8%. The drop is real and big enough to be worth the cost. Recommend sending SMS reminders in all clinics, after checking that SMS costs and patient complaints stay acceptable."

<b> Test B: discharge checklist (number outcome) </b>

<b> Step 1: Plan. </b> <br>
• H₀: average stay is the same with and without the checklist. <br>
• H₁: the averages are different. <br>
• Starting value (from records): average stay 6 days, standard deviation 3 days. Smallest change worth catching: 1 day shorter. The 0.05 line, power 80%.

In [5]:
from statsmodels.stats.power import TTestIndPower

# For numbers, the calculator needs: change worth catching ÷ standard deviation = 1 ÷ 3
n = TTestIndPower().solve_power(effect_size=1/3, alpha=0.05, power=0.8)
print(round(n))                             # prints 142 → we round up to 150 per group

142


Step 2: Get the data (practice data). 300 patients, put at random in the old-way group or the checklist group. Each patient gets a stay in days: on average 6 days for the old way and 5 days with the checklist, with the normal patient-to-patient variation (standard deviation 3).

In [6]:
rng = np.random.default_rng(4)              # 4 makes your numbers match these notes

patients_per_group = 150
group = np.array(["old_way"] * patients_per_group + ["checklist"] * patients_per_group)
rng.shuffle(group)                          # random assignment

# Each patient's expected stay: 5 days with the checklist, 6 days without
average_stay = np.where(group == "checklist", 5.0, 6.0)

# Real stays vary around the average (standard deviation 3), are whole days, and are at least 1 day
days = np.clip(np.round(rng.normal(average_stay, 3)), 1, None)

df = pd.DataFrame({"group": group, "days": days})
print(df.head())

       group  days
0    old_way  11.0
1  checklist   1.0
2  checklist   9.0
3  checklist   1.0
4  checklist   4.0


Step 3: Test.

In [7]:
# The t-test needs two lists of numbers: the stays of each group
old_way   = df.loc[df["group"] == "old_way", "days"]
checklist = df.loc[df["group"] == "checklist", "days"]

print(old_way.mean(), checklist.mean())          # 5.96  5.10
print(stats.ttest_ind(old_way, checklist).pvalue) # prints 0.010

# Cohen's d: how big is the difference compared with the normal spread?
combined_sd = ((old_way.std()**2 + checklist.std()**2) / 2) ** 0.5
print((old_way.mean() - checklist.mean()) / combined_sd)   # prints 0.30

5.96 5.1
0.010145462114409358
0.29876049660377324


Step 4: Decide. <br>
• p = 0.010, below 0.05, so we reject H₀. The checklist really shortens stays. <br>
• Size of the effect: 5.96 → 5.10 days, about 0.9 days shorter. Cohen's d = 0.30, a small-to-medium difference. <br>
• But: the manager said 1 full day was the smallest change worth it. 0.9 days is a little less. <br>
• Decision: "The checklist shortens hospital stays by about 0.9 days, and this is not chance. That is slightly less than the 1 day we planned for, so the manager should decide if it is still worth the staff time. With 150 beds, 0.9 days per patient still frees many bed-days each month."

This is a real lesson: a significant result is not automatically a "yes". You compare the size of the effect with what the manager said was worth it.